In [1]:
import subprocess, os, re
import numpy as np
import scipy.linalg
from Bio import Entrez, SeqIO
from itertools import product

PROJECT = r"C:\Users\Ganesh\Downloads\paml_project"
os.chdir(PROJECT)
os.makedirs("data",         exist_ok=True)
os.makedirs("codeml_runs",  exist_ok=True)
os.makedirs("evolver_runs", exist_ok=True)
os.makedirs("results",      exist_ok=True)

w = subprocess.run(['where', 'codeml'], capture_output=True, text=True)
print("codeml found:", w.returncode == 0)
print("codeml path: ", w.stdout.strip())
print("Working dir: ", os.getcwd())


codeml found: True
codeml path:  C:\Users\Ganesh\paml-4.10.10-win\bin\codeml.exe
Working dir:  C:\Users\Ganesh\Downloads\paml_project


In [2]:
Entrez.email = "tgiplayzfn@gmail.com"

GENE_NAME = "GAPDH"
accessions = {
    "human": "NM_002046",
    "mouse": "NM_008084",
    "rat":   "NM_017008",
    "dog":   "NM_001003142",
}

def fetch_cds(accession):
    handle = Entrez.efetch(
        db="nucleotide", id=accession,
        rettype="gb", retmode="text"
    )
    record = SeqIO.read(handle, "genbank")
    handle.close()
    for feat in record.features:
        if feat.type == "CDS":
            seq = feat.extract(record.seq)
            return str(seq[:len(seq) - len(seq) % 3])
    return str(record.seq)

sequences = {}
for sp, acc in accessions.items():
    print(f"Fetching {sp} ({acc})...")
    sequences[sp] = fetch_cds(acc)

print("\nLengths (nt):")
print({k: len(v) for k, v in sequences.items()})

Fetching human (NM_002046)...
Fetching mouse (NM_008084)...
Fetching rat (NM_017008)...
Fetching dog (NM_001003142)...

Lengths (nt):
{'human': 1008, 'mouse': 1002, 'rat': 1002, 'dog': 1002}


In [3]:
min_len = min(len(s) for s in sequences.values())
min_len = min_len - (min_len % 3)

sequences = {sp: seq[:min_len] for sp, seq in sequences.items()}

print(f"Trimmed to: {min_len} nt | {min_len // 3} codons")
print({k: len(v) for k, v in sequences.items()})

Trimmed to: 1002 nt | 334 codons
{'human': 1002, 'mouse': 1002, 'rat': 1002, 'dog': 1002}


In [4]:
def write_phylip(seqs, path):
    n, L = len(seqs), len(next(iter(seqs.values())))
    with open(path, "w") as f:
        f.write(f" {n}  {L}\n")
        for name, seq in seqs.items():
            f.write(f"{name[:10].ljust(10)}  {seq}\n")
    print(f"Written: {n} seqs x {L} nt ({L // 3} codons)")

write_phylip(sequences, os.path.join(PROJECT, "data", "alignment.phy"))

with open(os.path.join(PROJECT, "data", "tree.nwk"), "w") as f:
    f.write("((human, (mouse, rat)), dog);\n")

print("Tree written: ((human, (mouse, rat)), dog)")
print("alignment.phy exists:", os.path.exists(os.path.join(PROJECT, "data", "alignment.phy")))
print("tree.nwk exists:     ", os.path.exists(os.path.join(PROJECT, "data", "tree.nwk")))

Written: 4 seqs x 1002 nt (334 codons)
Tree written: ((human, (mouse, rat)), dog)
alignment.phy exists: True
tree.nwk exists:      True


In [6]:
# Fix tree file — PAML expects number of trees on first line
with open(os.path.join(PROJECT, "data", "tree.nwk"), "w") as f:
    f.write("1\n")
    f.write("((human, (mouse, rat)), dog);\n")

print("Tree file updated.")

Tree file updated.


In [7]:
ctl = (
    f"      seqfile = {PROJECT}\\data\\alignment.phy\n"
    f"     treefile = {PROJECT}\\data\\tree.nwk\n"
    f"      outfile = {PROJECT}\\codeml_runs\\codeml_out.txt\n"
    "        noisy = 3\n"
    "      verbose = 2\n"
    "      runmode = 0\n"
    "      seqtype = 1\n"
    "    CodonFreq = 3\n"
    "        model = 0\n"
    "      NSsites = 0\n"
    "        icode = 0\n"
    "    fix_kappa = 0\n"
    "        kappa = 2.0\n"
    "     fix_omega = 0\n"
    "        omega = 0.3\n"
    "    fix_alpha = 1\n"
    "        alpha = 0\n"
    "       getSE = 1\n"
)

ctl_path = os.path.join(PROJECT, "codeml_runs", "codeml.ctl")
with open(ctl_path, "w") as f:
    f.write(ctl)

result = subprocess.run(
    ["codeml", ctl_path],
    capture_output=True, text=True
)
print("Return code:", result.returncode)
print((result.stdout + result.stderr)[-800:])

Return code: 0
+C     2930.436996  0 0.0016   747 | 2/8
 44 h-m-p  0.0006 0.0090   1.6593 +++    2930.434546  m 0.0090   765 | 3/8
 45 h-m-p  0.0137 0.3746   0.8264 --C    2930.434515  0 0.0003   778 | 3/8
 46 h-m-p  0.0160 8.0000   0.0552 ++YC   2930.433494  1 0.1899   797 | 3/8
 47 h-m-p  0.3816 8.0000   0.0275 YC     2930.433328  1 0.1992   814 | 3/8
 48 h-m-p  1.6000 8.0000   0.0000 Y      2930.433326  0 0.9940   830 | 3/8
 49 h-m-p  1.6000 8.0000   0.0000 Y      2930.433326  0 1.2397   846 | 3/8
 50 h-m-p  1.6000 8.0000   0.0000 Y      2930.433326  0 1.6000   862 | 3/8
 51 h-m-p  1.6000 8.0000   0.0000 C      2930.433326  0 1.6000   878
Out..
lnL  = -2930.433326
879 lfun, 879 eigenQcodon, 5274 P(t)
Calculating SE's


Time used:  0:03



In [8]:
out_path = os.path.join(PROJECT, "codeml_runs", "codeml_out.txt")
with open(out_path) as f:
    text = f.read()

kappa = float(re.search(r"kappa \(ts/tv\)\s*=\s*([\d.]+)", text).group(1))
omega = float(re.search(r"omega \(dN/dS\)\s*=\s*([\d.]+)", text).group(1))
lnL   = float(re.search(r"lnL\(ntime.*?\):\s*([-\d.]+)", text).group(1))

freq_m = re.search(r"Codon frequencies under model.*?\n([\s\S]+?)\n\n\n", text)
pi = [float(x) for x in re.findall(r"[\d.]+", freq_m.group(1))]

ALL64  = [''.join(t) for t in product(['T','C','A','G'], repeat=3)]
pi_all = dict(zip(ALL64, pi))

print(f"kappa = {kappa:.4f}")
print(f"omega = {omega:.4f}")
print(f"lnL   = {lnL:.2f}")
print(f"pi    = {len(pi)} values, sum = {sum(pi):.6f}")

kappa = 1.8425
omega = 0.0599
lnL   = -2930.43
pi    = 64 values, sum = 1.000000


In [9]:
GENETIC_CODE = {
    'TTT':'F','TTC':'F','TTA':'L','TTG':'L','CTT':'L','CTC':'L',
    'CTA':'L','CTG':'L','ATT':'I','ATC':'I','ATA':'I','ATG':'M',
    'GTT':'V','GTC':'V','GTA':'V','GTG':'V','TCT':'S','TCC':'S',
    'TCA':'S','TCG':'S','CCT':'P','CCC':'P','CCA':'P','CCG':'P',
    'ACT':'T','ACC':'T','ACA':'T','ACG':'T','GCT':'A','GCC':'A',
    'GCA':'A','GCG':'A','TAT':'Y','TAC':'Y','TAA':'*','TAG':'*',
    'CAT':'H','CAC':'H','CAA':'Q','CAG':'Q','AAT':'N','AAC':'N',
    'AAA':'K','AAG':'K','GAT':'D','GAC':'D','GAA':'E','GAG':'E',
    'TGT':'C','TGC':'C','TGA':'*','TGG':'W','CGT':'R','CGC':'R',
    'CGA':'R','CGG':'R','AGT':'S','AGC':'S','AGA':'R','AGG':'R',
    'GGT':'G','GGC':'G','GGA':'G','GGG':'G',
}
NTS   = ['T','C','A','G']
TS    = {('T','C'),('C','T'),('A','G'),('G','A')}
SENSE = [c for c in ALL64 if GENETIC_CODE.get(c,'*') != '*']
N     = len(SENSE)

pi61  = [pi_all[c] for c in SENSE]

def build_Q(kappa, omega, pi):
    Q = np.zeros((N, N))
    for i, ci in enumerate(SENSE):
        for j, cj in enumerate(SENSE):
            if i == j:
                continue
            d = [(a, b) for a, b in zip(ci, cj) if a != b]
            if len(d) != 1:
                continue
            a, b = d[0]
            rate = pi[j]
            if (a, b) in TS:
                rate *= kappa
            if GENETIC_CODE[ci] != GENETIC_CODE[cj]:
                rate *= omega
            Q[i, j] = rate
    for i in range(N):
        Q[i, i] = -Q[i, :].sum()
    mu = -sum(pi[i] * Q[i, i] for i in range(N))
    return Q / mu

Q  = build_Q(kappa, omega, pi61)
t  = 0.1
Pt = scipy.linalg.expm(Q * t)

np.save(os.path.join(PROJECT, "results", "q_matrix.npy"), Q)

print(f"Q shape:       {Q.shape}")
print(f"P(t) row sum:  {Pt[0].sum():.8f}")
print(f"Q saved to:    results\\q_matrix.npy")

Q shape:       (61, 61)
P(t) row sum:  1.00000000
Q saved to:    results\q_matrix.npy


In [10]:
omega_sim = min(omega, 5.0)

pi_block_rows = []
for i in range(0, 64, 4):
    row = "  ".join(f"{pi[i+j]:.8f}" for j in range(4))
    pi_block_rows.append(row)
pi_block = "\n".join(pi_block_rows)

n_codons  = min_len // 3
evolver_tree = "((human:0.05, (mouse:0.10, rat:0.08):0.12):0.15, dog:0.20);"

mccodon = (
    "0          * 0: paml format output\n"
    "13147      * random number seed\n"
    f"4 {n_codons} 1    * nseqs  ncodons  nreplicates\n"
    "-1         * use absolute branch lengths\n"
    f"{evolver_tree}\n"
    f"{omega_sim:.4f}     * omega\n"
    f"{kappa:.4f}     * kappa\n"
    f"{pi_block}\n"
    "0          * genetic code (0: universal)\n"
)

mccodon_path = os.path.join(PROJECT, "evolver_runs", "MCcodon.dat")
with open(mccodon_path, "w") as f:
    f.write(mccodon)

res = subprocess.run(
    ["evolver", "6"],
    input='',
    cwd=os.path.join(PROJECT, "evolver_runs"),
    capture_output=True, text=True,
    timeout=60
)
print("Return code:", res.returncode)
print((res.stdout + res.stderr)[-500:])
print("\nFiles in evolver_runs:")
print(os.listdir(os.path.join(PROJECT, "evolver_runs")))

Return code: 0
2252 0.020270
 0.011261 0.022523 0.003003 0.000000
 0.010511 0.013514 0.000751 0.018018
 0.011261 0.006006 0.003754 0.000751
 0.018018 0.046547 0.000000 0.030781
 0.020270 0.035285 0.005255 0.000751
 0.014264 0.045045 0.021021 0.056306
 0.001502 0.009009 0.000000 0.008258
 0.009009 0.042793 0.002252 0.045045
 0.036787 0.051051 0.006006 0.001502
 0.021772 0.038288 0.006006 0.033784
 0.014264 0.045045 0.010511 0.026276
genetic code: 0

All parameters are read.  Ready to simulate


did data set 1 


Files in evolver_runs:
['ancestral.txt', 'evolve.ctl', 'evolver.out', 'mc.aa.txt', 'mc.codon12.txt', 'mc.txt', 'MCcodon.dat']


In [11]:
def read_mc_txt(path):
    seqs = {}
    with open(path) as f:
        header = ""
        while not header.strip():
            header = f.readline()
        n, L_nt = int(header.split()[0]), int(header.split()[1])
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            name  = parts[0]
            seq   = "".join(parts[1:])
            if seq:
                seqs[name] = seq
    print(f"Evolved: {len(seqs)} sequences x {L_nt} nt ({L_nt // 3} codons)")
    return seqs

mc_path = os.path.join(PROJECT, "evolver_runs", "mc.txt")
evolved  = read_mc_txt(mc_path)

for sp, seq in evolved.items():
    codons = [seq[i:i+3] for i in range(0, len(seq), 3)]
    print(f"  {sp:10s}  {' '.join(codons[:8])}...")

print(f"\nPipeline complete.")
print(f"  Gene:   {GENE_NAME} — human, mouse, rat, dog")
print(f"  kappa = {kappa:.4f}")
print(f"  omega = {omega:.4f}")
print(f"  lnL   = {lnL:.2f}")
print(f"  Q     = {Q.shape} matrix saved to results\\q_matrix.npy")

Evolved: 4 sequences x 1002 nt (334 codons)
  human       CCC GTC ACC GCT TGG AAC CTC AAA...
  mouse       CCC GTC ACC GCC TGG AAC CTC AAA...
  rat         CCC GTC ACC GCT TGG AAC CTC AAA...
  dog         CCC GTG ACC GCT TGG AAC CTG AAG...

Pipeline complete.
  Gene:   GAPDH — human, mouse, rat, dog
  kappa = 1.8425
  omega = 0.0599
  lnL   = -2930.43
  Q     = (61, 61) matrix saved to results\q_matrix.npy


In [12]:
print("=" * 65)
print(f"INITIAL GENE SEQUENCES  |  {GENE_NAME}")
print("=" * 65)

for sp, seq in sequences.items():
    codons = [seq[i:i+3] for i in range(0, len(seq), 3)]
    print(f"\n{sp.upper()}  ({accessions[sp]})")
    print(f"  Length : {len(seq)} nt  |  {len(codons)} codons")
    print(f"  First 10 codons : {' '.join(codons[:10])}")
    print(f"  Last  10 codons : {' '.join(codons[-10:])}")
    print(f"  Full nucleotide sequence:")
    for i in range(0, len(seq), 60):
        print(f"    {seq[i:i+60]}")

INITIAL GENE SEQUENCES  |  GAPDH

HUMAN  (NM_002046)
  Length : 1002 nt  |  334 codons
  First 10 codons : ATG GGG AAG GTG AAG GTC GGA GTC AAC GGA
  Last  10 codons : GTG GAC CTC ATG GCC CAC ATG GCC TCC AAG
  Full nucleotide sequence:
    ATGGGGAAGGTGAAGGTCGGAGTCAACGGATTTGGTCGTATTGGGCGCCTGGTCACCAGG
    GCTGCTTTTAACTCTGGTAAAGTGGATATTGTTGCCATCAATGACCCCTTCATTGACCTC
    AACTACATGGTTTACATGTTCCAATATGATTCCACCCATGGCAAATTCCATGGCACCGTC
    AAGGCTGAGAACGGGAAGCTTGTCATCAATGGAAATCCCATCACCATCTTCCAGGAGCGA
    GATCCCTCCAAAATCAAGTGGGGCGATGCTGGCGCTGAGTACGTCGTGGAGTCCACTGGC
    GTCTTCACCACCATGGAGAAGGCTGGGGCTCATTTGCAGGGGGGAGCCAAAAGGGTCATC
    ATCTCTGCCCCCTCTGCTGATGCCCCCATGTTCGTCATGGGTGTGAACCATGAGAAGTAT
    GACAACAGCCTCAAGATCATCAGCAATGCCTCCTGCACCACCAACTGCTTAGCACCCCTG
    GCCAAGGTCATCCATGACAACTTTGGTATCGTGGAAGGACTCATGACCACAGTCCATGCC
    ATCACTGCCACCCAGAAGACTGTGGATGGCCCCTCCGGGAAACTGTGGCGTGATGGCCGC
    GGGGCTCTCCAGAACATCATCCCTGCCTCTACTGGCGCTGCCAAGGCTGTGGGCAAGGTC
    ATCCCTGAGCTGAACGGGAAGCTCACTGGCATGGCCTTCCGTGTCC

In [13]:
print("=" * 65)
print("MODEL PARAMETERS")
print("=" * 65)
print(f"  kappa  (transition / transversion ratio) = {kappa:.4f}")
print(f"  omega  (dN/dS nonsynonymous ratio)       = {omega:.4f}")
print(f"  lnL    (log-likelihood of model fit)     = {lnL:.4f}")
print(f"  Codons in alignment                      = {min_len // 3}")

print()
print("=" * 65)
print("Q MATRIX  (61 x 61  GY94 rate matrix)")
print("=" * 65)
print(f"  Shape                    : {Q.shape}")
print(f"  Non-zero off-diagonal    : {np.count_nonzero(Q - np.diag(np.diag(Q)))}")
print(f"  Min off-diagonal rate    : {Q[Q > 0].min():.6f}")
print(f"  Max off-diagonal rate    : {Q[Q > 0].max():.6f}")
print(f"  Row sum range (should ~0): {Q.sum(axis=1).min():.2e}  to  {Q.sum(axis=1).max():.2e}")
print()
print("Codon labels — first 10 rows / cols:")
print(SENSE[:10])
print()
print("First 10 x 10 block of Q:")
print(np.array2string(Q[:10, :10], precision=5, suppress_small=True))
print()
print("Full Q matrix:")
print(np.array2string(Q, precision=5, suppress_small=True))

MODEL PARAMETERS
  kappa  (transition / transversion ratio) = 1.8425
  omega  (dN/dS nonsynonymous ratio)       = 0.0599
  lnL    (log-likelihood of model fit)     = -2930.4333
  Codons in alignment                      = 334

Q MATRIX  (61 x 61  GY94 rate matrix)
  Shape                    : (61, 61)
  Non-zero off-diagonal    : 500
  Min off-diagonal rate    : 0.000755
  Max off-diagonal rate    : 1.742835
  Row sum range (should ~0): -2.22e-16  to  4.15e-16

Codon labels — first 10 rows / cols:
['TTT', 'TTC', 'TTA', 'TTG', 'TCT', 'TCC', 'TCA', 'TCG', 'TAT', 'TAC']

First 10 x 10 block of Q:
[[-0.85083  0.76685  0.00227  0.00378  0.02922  0.       0.       0.
   0.0151   0.     ]
 [ 0.46476 -0.68649  0.00227  0.00378  0.       0.05704  0.       0.
   0.       0.0151 ]
 [ 0.0151   0.02492 -0.23236  0.11619  0.       0.       0.00417  0.
   0.       0.     ]
 [ 0.0151   0.02492  0.06971 -0.82526  0.       0.       0.       0.00278
   0.       0.     ]
 [ 0.02782  0.       0.       0.  

In [14]:
print("=" * 65)
print("EVOLVED GENE SEQUENCES  (evolver Monte Carlo simulation)")
print("=" * 65)
print(f"  kappa used : {kappa:.4f}")
print(f"  omega used : {omega_sim:.4f}{'  (capped)' if omega_sim < omega else ''}")
print(f"  Tree       : {evolver_tree}")

for sp, seq in evolved.items():
    codons      = [seq[i:i+3] for i in range(0, len(seq), 3)]
    orig_seq    = sequences.get(sp, seq)
    orig_codons = [orig_seq[i:i+3] for i in range(0, len(orig_seq), 3)]
    n_diff      = sum(1 for a, b in zip(codons, orig_codons) if a != b)

    print(f"\n{sp.upper()}")
    print(f"  Length          : {len(seq)} nt  |  {len(codons)} codons")
    print(f"  Codon changes   : {n_diff} / {len(codons)}  ({100*n_diff/len(codons):.1f}% diverged from input)")
    print(f"  First 10 codons : {' '.join(codons[:10])}")
    print(f"  Last  10 codons : {' '.join(codons[-10:])}")
    print(f"  Full nucleotide sequence:")
    for i in range(0, len(seq), 60):
        print(f"    {seq[i:i+60]}")

EVOLVED GENE SEQUENCES  (evolver Monte Carlo simulation)
  kappa used : 1.8425
  omega used : 0.0599
  Tree       : ((human:0.05, (mouse:0.10, rat:0.08):0.12):0.15, dog:0.20);

HUMAN
  Length          : 1002 nt  |  334 codons
  Codon changes   : 325 / 334  (97.3% diverged from input)
  First 10 codons : CCC GTC ACC GCT TGG AAC CTC AAA ACA TGG
  Last  10 codons : GGC TGG ACC GGG GTG AGG AAC TCT TTG AGC
  Full nucleotide sequence:
    CCCGTCACCGCTTGGAACCTCAAAACATGGTTCGTGTCCATCTTCGCCCCCGATACCGGG
    CACATCGACTTTTCTATGTCCGAGGACATCGGACCTAACGCCAAGGTCGCTGGACTGCCC
    GCCGTGGCCTCACTCTGCGGCGACGTGGCCGCTCTCTATACCGCCAAAGCTTTCTGTGCC
    CTCGCTGGCTCTGGCGGCATCGAGATCGCCCTGTGCCCCGACAGGGCTCGTATGGTCGCG
    TTCGTCCAGGGTACTGCTCCTCCAGCTCTTATCACTGCTGCCAAGGATGCTGGCTACGGG
    TCTCTGTCCGCTCGACCCCCCATGATCGCCCCTCAGGAGGATAATGCCATCCGTGTGGCC
    GACGCCCACGCTCTGAAGCCCTATGGGCCTGGTGTGAACATCACCGCTGGCGATCAGGTC
    ATCCGCGTGAAAGTGGTCAAGACTAAGGCTTCTTCAAAGAACGCCAAGACTATGCTGATC
    GCTACTGCTCCTTCCGGGCCCGTGAAGTACGGCGGGAATTATT